# 🚀 IAI.tv to Google Drive Cloud-to-Cloud Downloader
### 100% Cloud-to-Cloud Transfer (Zero Local Mac Disk Space & Zero Home Internet Data)

This notebook runs entirely on Google's Cloud datacenter servers (1 Gbps speed).
It downloads full HD debates and talks directly from **IAI.tv's CDN** straight into your **Google Drive**.

> **⚡ Highlights:**
> - **0 MB local Mac storage used:** Videos go straight into Google Drive.
> - **0 MB home Wi-Fi used:** Download happens on Google's cloud backbone.
> - **Smart Resume:** If you stop and restart, it skips already downloaded videos.
> - **750 GB Daily Quota Safe:** Google Drive allows max 750 GB upload/day (~450-500 videos/day). Run ~450 videos daily for 4 days to archive all 2,050 videos into your 5 TB Drive!

---

## Step 1: Mount your Google Drive
Click the **Play button ▶** on the cell below. Google will ask for permission to access your Google Drive. Grant access so Colab can save videos into `MyDrive/IAI_Videos`.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define target directory inside Google Drive
DEST_DIR = '/content/drive/MyDrive/IAI_Videos'
os.makedirs(DEST_DIR, exist_ok=True)
print(f"✅ Target Google Drive folder ready: {DEST_DIR}")

## Step 2: Download Configuration
Choose how many videos you want to download in this session.
Recommended: **450 videos per day** to stay comfortably within Google Drive's 750 GB/day upload quota.

In [ ]:
# Configuration settings
BATCH_LIMIT = 450         # Max videos to download today (Recommended: 450)
MAX_GB_TODAY = 700        # Stop if downloaded GB reaches this limit (Safeguard for 750 GB/day)
CHUNK_SIZE = 4 * 1024 * 1024  # 4 MB streaming buffer for ultra-fast writing

print(f"Settings active: BATCH_LIMIT={BATCH_LIMIT}, MAX_GB_TODAY={MAX_GB_TODAY} GB")

## Step 3: Fetch & Load Video Catalog (2,050 Videos)
Click **Play ▶** to load the complete index of all 2,050 videos directly from IAI.tv.

In [ ]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import json
import ssl
import re
from pathlib import Path

CATALOG_FILE = os.path.join(DEST_DIR, 'iai_videos_catalog.json')
ctx = ssl._create_unverified_context()
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

def sanitize_filename(name):
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    return name.strip()

if os.path.exists(CATALOG_FILE):
    print(f"📂 Loading existing catalog from {CATALOG_FILE}...")
    with open(CATALOG_FILE, 'r', encoding='utf-8') as f:
        video_catalog = json.load(f)
    print(f"✅ Loaded {len(video_catalog)} videos from cached catalog.")
else:
    print("🌐 Fetching fresh video sitemap from https://iai.tv/VideoController/videoSitemap.xml...")
    req = urllib.request.Request('https://iai.tv/VideoController/videoSitemap.xml', headers=headers)
    with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
        xml_data = resp.read()
    
    root = ET.fromstring(xml_data)
    ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9', 'video': 'http://www.google.com/schemas/sitemap-video/1.1'}
    urls = root.findall('sm:url', ns)
    
    video_catalog = []
    seen_fn = {}
    for idx, u in enumerate(urls, 1):
        loc = u.find('sm:loc', ns).text.strip() if u.find('sm:loc', ns) is not None else ''
        slug = loc.split('/')[-1] if loc else f'video-{idx}'
        v = u.find('video:video', ns)
        title = v.find('video:title', ns).text.strip() if (v is not None and v.find('video:title', ns) is not None and v.find('video:title', ns).text) else slug
        content_loc = v.find('video:content_loc', ns).text.strip() if (v is not None and v.find('video:content_loc', ns) is not None and v.find('video:content_loc', ns).text) else ''
        dur = v.find('video:duration', ns).text.strip() if (v is not None and v.find('video:duration', ns) is not None and v.find('video:duration', ns).text) else '0'
        pub = v.find('video:publication_date', ns).text.strip() if (v is not None and v.find('video:publication_date', ns) is not None and v.find('video:publication_date', ns).text) else ''
        
        if content_loc and content_loc.endswith('.mp4'):
            raw_fn = urllib.parse.unquote(content_loc.split('/')[-1])
            fn = sanitize_filename(raw_fn)
        else:
            fn = sanitize_filename(f"{slug}.mp4")
            
        if fn in seen_fn:
            seen_fn[fn] += 1
            fn = f"{fn[:-4]}_{seen_fn[fn]}.mp4"
        else:
            seen_fn[fn] = 1
            
        video_catalog.append({
            'id': idx,
            'title': title,
            'page_url': loc,
            'video_url': content_loc,
            'duration_sec': int(dur) if dur.isdigit() else 0,
            'publication_date': pub,
            'filename': fn
        })
    
    with open(CATALOG_FILE, 'w', encoding='utf-8') as f:
        json.dump(video_catalog, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved fresh catalog with {len(video_catalog)} videos to {CATALOG_FILE}.")

## Step 4: Start Cloud-to-Cloud Transfer ▶
Click **Play ▶** below to begin transferring videos.
- Videos are streamed at 50-100 MB/s directly into `MyDrive/IAI_Videos/`.
- Files already on Google Drive are automatically skipped.
- Progress is logged live with download speed and remaining count.

In [ ]:
import time
import sys

HISTORY_FILE = os.path.join(DEST_DIR, 'download_history.txt')
completed = set()
if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            completed.add(line.strip())

print(f"📊 Existing completed downloads in history: {len(completed)}")

downloaded_in_session = 0
total_bytes_session = 0

for item in video_catalog:
    if downloaded_in_session >= BATCH_LIMIT:
        print(f"\n🎯 Reached batch limit of {BATCH_LIMIT} videos for today! Stopping.")
        break
        
    if (total_bytes_session / (1024**3)) >= MAX_GB_TODAY:
        print(f"\n⚠️ Reached {MAX_GB_TODAY} GB daily safety limit! Stopping to protect daily quota.")
        break
        
    fn = item['filename']
    vurl = item['video_url']
    
    if not vurl or not vurl.startswith('http'):
        continue
        
    target_path = os.path.join(DEST_DIR, fn)
    
    # Skip if file already exists and has size > 1MB
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        completed.add(fn)
        continue
        
    encoded_url = urllib.parse.quote(vurl, safe=':/?&=')
    print(f"\n[{downloaded_in_session + 1}/{BATCH_LIMIT}] 📥 Downloading: {item['title'][:65]}...")
    print(f"    Target: {fn}")
    
    try:
        req = urllib.request.Request(encoded_url, headers=headers)
        start_time = time.time()
        with urllib.request.urlopen(req, context=ctx, timeout=60) as resp:
            content_len = resp.headers.get('Content-Length')
            total_size = int(content_len) if content_len and content_len.isdigit() else 0
            total_mb = total_size / (1024 * 1024) if total_size else 0
            
            temp_path = target_path + ".part"
            downloaded_bytes = 0
            last_print = time.time()
            
            with open(temp_path, 'wb') as out_f:
                while True:
                    chunk = resp.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    out_f.write(chunk)
                    downloaded_bytes += len(chunk)
                    
                    now = time.time()
                    if now - last_print > 1.5 or (total_size and downloaded_bytes == total_size):
                        elapsed = max(now - start_time, 0.001)
                        speed_mb = (downloaded_bytes / (1024 * 1024)) / elapsed
                        cur_mb = downloaded_bytes / (1024 * 1024)
                        pct = (downloaded_bytes / total_size * 100) if total_size else 0
                        sys.stdout.write(f"\r    ⚡ {cur_mb:.1f}/{total_mb:.1f} MB [{pct:5.1f}%] @ {speed_mb:5.1f} MB/s")
                        sys.stdout.flush()
                        last_print = now
            
            os.replace(temp_path, target_path)
            downloaded_in_session += 1
            total_bytes_session += downloaded_bytes
            completed.add(fn)
            
            with open(HISTORY_FILE, 'a', encoding='utf-8') as hf:
                hf.write(f"{fn}\n")
                
            elapsed_total = max(time.time() - start_time, 0.1)
            avg_speed = (downloaded_bytes / (1024 * 1024)) / elapsed_total
            print(f"\n    ✅ Finished in {elapsed_total:.1f}s (Avg {avg_speed:.1f} MB/s)")
            
    except KeyboardInterrupt:
        print("\n🛑 Paused by user.")
        break
    except Exception as err:
        print(f"\n❌ Error downloading {fn}: {err}")
        if os.path.exists(target_path + ".part"):
            try:
                os.remove(target_path + ".part")
            except:
                pass
        time.sleep(2)

total_gb_session = total_bytes_session / (1024**3)
print("\n" + "="*60)
print(f"🎉 Session Complete!")
print(f"• Videos downloaded in this run: {downloaded_in_session}")
print(f"• Data transferred directly to Google Drive: {total_gb_session:.2f} GB")
print(f"• Total completed in Google Drive: {len(completed)} / {len(video_catalog)}")
print(f"• Local Mac disk space used: 0.00 MB")
print("="*60)